# 02. Basic RAG Pipeline

**Topics covered:** Retrieval + Generation · Prompt Engineering for RAG

This notebook builds on [01_chunking_embeddings_vectorstore.ipynb](https://github.com/S33mi/modern-ai-llm-journey/blob/main/04_rag_systems/01_chunking_embeddings_vectorstore.ipynb).

We will:
1. Detect **GPU vs CPU** and pick suitable models automatically
2. Chunk documents, embed them, and build a vector index
3. Retrieve top-k passages for a query
4. Design a solid **RAG prompt**
5. Generate an answer grounded in the retrieved context
6. Run the full pipeline end-to-end

## 1. Setup & Device Detection

```bash
pip install transformers sentence-transformers faiss-cpu chromadb langchain-text-splitters accelerate
```

> On GPU Colab you can also install `faiss-gpu` if you prefer.

In [32]:
# ! pip install transformers sentence-transformers faiss-cpu chromadb langchain-text-splitters accelerate
# pip install transformers sentence-transformers faiss-gpu chromadb langchain-text-splitters accelerate

In [33]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU – using small models for generation.")

Device: cpu
Running on CPU – using small models for generation.


## 2. Model Choices (GPU-first, CPU fallback)

| Role | GPU | CPU |
|------|-----|-----|
| Embeddings | `all-MiniLM-L6-v2` | same (fast on CPU) |
| Generator | `google/flan-t5-base` or small causal LM | `google/flan-t5-small` / `sshleifer/tiny-gpt2` |

We use **FLAN-T5** for generation when possible: it is instruction-tuned and works well for short grounded answers even at small size.

In [34]:
# Embedding model (good on both CPU and GPU)
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Generator: prefer a slightly larger model on GPU
if DEVICE == "cuda":
    GEN_MODEL_NAME = "google/flan-t5-base"   # ~250M, solid quality
else:
    GEN_MODEL_NAME = "google/flan-t5-small"  # ~80M, CPU-friendly

print(f"Embed model : {EMBED_MODEL_NAME}")
print(f"Generator   : {GEN_MODEL_NAME}")

Embed model : sentence-transformers/all-MiniLM-L6-v2
Generator   : google/flan-t5-small


## 3. Sample Knowledge Base

Short notes so retrieval and generation stay fast on CPU.

In [35]:
documents = [
    {
        "id": "doc1",
        "title": "Attention Mechanisms",
        "text": (
            "Scaled dot-product attention computes similarity between queries and keys, "
            "scales by the square root of the key dimension, applies softmax, and uses "
            "the resulting weights to combine values. Multi-head attention runs several "
            "attention heads in parallel so the model can capture different types of "
            "relationships. Causal masking prevents tokens from attending to future "
            "positions, which is essential for autoregressive language models like GPT."
        ),
    },
    {
        "id": "doc2",
        "title": "Transformers and Residuals",
        "text": (
            "A Transformer block typically contains multi-head self-attention followed by "
            "a position-wise feed-forward network. Residual connections and layer "
            "normalization stabilize training of deep stacks. Pre-LN is the dominant design "
            "in modern LLMs such as GPT-2, LLaMA, and Mistral. The feed-forward network "
            "usually expands the hidden size by a factor of four and uses GELU or SwiGLU."
        ),
    },
    {
        "id": "doc3",
        "title": "Fine-Tuning and LoRA",
        "text": (
            "Full fine-tuning updates every parameter of a pre-trained model. This is "
            "expensive in memory and storage. LoRA freezes the base weights and injects "
            "small trainable low-rank matrices. The rank r controls capacity; alpha scales "
            "the update. QLoRA combines 4-bit quantization of the base model with LoRA "
            "adapters so that 7B to 13B models can be fine-tuned on a single consumer GPU."
        ),
    },
    {
        "id": "doc4",
        "title": "Embeddings and Similarity",
        "text": (
            "Sentence embeddings map text into a dense vector space where semantically "
            "similar sentences lie close together. Cosine similarity is the standard "
            "metric. Models such as all-MiniLM-L6-v2 and bge-small produce strong "
            "embeddings for retrieval. Mean pooling of the last hidden states is a common "
            "way to obtain a fixed-size sentence vector from a Transformer."
        ),
    },
    {
        "id": "doc5",
        "title": "RAG Overview",
        "text": (
            "Retrieval-Augmented Generation (RAG) combines a retriever with a generator. "
            "Documents are chunked and embedded into a vector store. At query time the "
            "system retrieves the most relevant chunks and passes them as context to an "
            "LLM, which produces an answer grounded in that context. Good chunking, "
            "strong embeddings, and careful prompt design all improve RAG quality."
        ),
    },
]

print(f"{len(documents)} documents loaded.")

5 documents loaded.


## 4. Chunk → Embed → Index

In [36]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for doc in documents:
    parts = splitter.split_text(doc["text"])
    for i, part in enumerate(parts):
        chunks.append({
            "chunk_id": f"{doc['id']}_c{i}",
            "doc_id": doc["id"],
            "title": doc["title"],
            "text": part.strip(),
        })

print(f"Total chunks: {len(chunks)}")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
chunk_texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.asarray(embeddings, dtype="float32"))
print(f"FAISS index: {index.ntotal} vectors, dim={dim}")

Total chunks: 16


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index: 16 vectors, dim=384


## 5. Retriever

In [37]:
def retrieve(query: str, k: int = 3):
    """Return top-k chunks with scores."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.asarray(q_emb, dtype="float32"), k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        c = chunks[int(idx)]
        results.append({
            "score": float(score),
            "title": c["title"],
            "text": c["text"],
            "chunk_id": c["chunk_id"],
        })
    return results


# Quick test
for h in retrieve("What is LoRA?", k=2):
    print(f"[{h['score']:.3f}] {h['title']}: {h['text'][:100]}...")

[0.561] Fine-Tuning and LoRA: . LoRA freezes the base weights and injects small trainable low-rank matrices. The rank r controls c...
[0.360] Fine-Tuning and LoRA: . QLoRA combines 4-bit quantization of the base model with LoRA adapters so that 7B to 13B models ca...


## 6. Load the Generator

In [59]:
# from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)

# FLAN-T5 is seq2seq – use the text2text pipeline (works on CPU and GPU)
#generator = pipeline(
#    "text2text-generation", #text2text-generation is no longer supported in updated transformers
#    model=GEN_MODEL_NAME,
#    tokenizer=tokenizer,
#    device=0 if DEVICE == "cuda" else -1,
#    max_new_tokens=128,
#)

# print(f"Generator ready on device={'cuda:0' if DEVICE == 'cuda' else 'cpu'}")


In [50]:
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(DEVICE)

def generate(prompt, max_new_tokens=128, do_sample=False, temperature=0.7, **kwargs):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            num_beams=4 if not do_sample else 1,
            early_stopping=True,
            **kwargs
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Generator ready on device={DEVICE}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generator ready on device=cpu


## 7. Prompt Engineering for RAG

A good RAG prompt usually includes:

1. **Role / instruction** – answer only from the context
2. **Retrieved context** – clearly delimited
3. **User question**
4. Optional: ask for “I don’t know” when the context is insufficient

```
Use only the following context to answer the question.
If the answer is not in the context, say "I don't know based on the provided context."

Context:
[chunk 1]
[chunk 2]
...

Question: ...
Answer:
```

In [51]:
def build_rag_prompt(query: str, retrieved: list) -> str:
    context_blocks = []
    for i, r in enumerate(retrieved, 1):
        context_blocks.append(f"[{i}] ({r['title']}) {r['text']}")
    context = "\n\n".join(context_blocks)

    prompt = (
        "Use only the following context to answer the question. "
        "If the answer is not in the context, say "
        "\"I don't know based on the provided context.\"\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return prompt


# Preview
demo_hits = retrieve("What is multi-head attention?", k=2)
print(build_rag_prompt("What is multi-head attention?", demo_hits)[:500], "...")

Use only the following context to answer the question. If the answer is not in the context, say "I don't know based on the provided context."

Context:
[1] (Attention Mechanisms) . Multi-head attention runs several attention heads in parallel so the model can capture different types of relationships

[2] (Transformers and Residuals) A Transformer block typically contains multi-head self-attention followed by a position-wise feed-forward network

Question: What is multi-head attention?
Answer: ...


## 8. Full RAG Pipeline

In [54]:
def rag_answer(query: str, k: int = 3, show_context: bool = True) -> str:
    """Retrieve → prompt → generate."""
    hits = retrieve(query, k=k)

    if show_context:
        print("Retrieved context:")
        for h in hits:
            print(f"  [{h['score']:.3f}] {h['title']}: {h['text'][:90]}...")
        print()

    prompt = build_rag_prompt(query, hits)
    #out = generator(prompt, max_new_tokens=100, do_sample=False)
    out = generate(prompt, max_new_tokens=100, do_sample=False)
    #answer = out[0]["generated_text"].strip()
    answer = out.strip()
    return answer


print("Pipeline ready.")

Pipeline ready.


## 9. Try It

In [55]:
questions = [
    "What is multi-head attention and why is it used?",
    "Explain LoRA in simple terms.",
    "How does RAG work?",
    "What activation is often used in the Transformer feed-forward network?",
    "Who invented the airplane?",   # should trigger "I don't know" style answer
]

for q in questions:
    print("=" * 60)
    print(f"Q: {q}")
    ans = rag_answer(q, k=3, show_context=True)
    print(f"A: {ans}\n")

Q: What is multi-head attention and why is it used?
Retrieved context:
  [0.721] Attention Mechanisms: . Multi-head attention runs several attention heads in parallel so the model can capture d...
  [0.581] Transformers and Residuals: A Transformer block typically contains multi-head self-attention followed by a position-wi...
  [0.304] Attention Mechanisms: Scaled dot-product attention computes similarity between queries and keys, scales by the s...

A: the model can capture different types of relationships

Q: Explain LoRA in simple terms.
Retrieved context:
  [0.549] Fine-Tuning and LoRA: . LoRA freezes the base weights and injects small trainable low-rank matrices. The rank r ...
  [0.323] Fine-Tuning and LoRA: . QLoRA combines 4-bit quantization of the base model with LoRA adapters so that 7B to 13B...
  [0.155] RAG Overview: . At query time the system retrieves the most relevant chunks and passes them as context t...

A: [3]

Q: How does RAG work?
Retrieved context:
  [0.578] RAG

## 10. Prompt Variants (Experiment)

Small changes in wording often change answer quality. Try these patterns:

In [58]:
def build_prompt_variant(query, retrieved, style="strict"):
    context = "\n".join(f"- {r['text']}" for r in retrieved)

    if style == "strict":
        return (
            f"Answer ONLY using the context below. "
            f"If missing, reply exactly: I don't know.\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        )
    if style == "cite":
        return (
            f"Answer the question using the context. "
            f"Mention the source title when possible.\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        )
    # default conversational
    return (
        f"You are a helpful assistant. Use the context to answer.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )


q = "What is QLoRA?"
hits = retrieve(q, k=2)
for style in ["strict", "cite", "conversational"]:
    prompt = build_prompt_variant(q, hits, style=style)
    #ans = generator(prompt, max_new_tokens=80, do_sample=False)[0]["generated_text"].strip()
    ans = generate(prompt, max_new_tokens=80, do_sample=False).strip()
    print(f"[{style}]\n{ans}\n")

[strict]
combines 4-bit quantization of the base model

[cite]
QLoRA

[conversational]
combines 4-bit quantization of the base model with LoRA adapters



## 11. Summary

| Stage | What happens |
|-------|--------------|
| **Chunk** | Split docs with overlap |
| **Embed** | Dense vectors (MiniLM) |
| **Index** | FAISS inner-product search |
| **Retrieve** | Top-k chunks for the query |
| **Prompt** | Instruction + context + question |
| **Generate** | FLAN-T5 (or other LM) produces the answer |

### Device strategy used in this notebook

```text
if GPU available → flan-t5-base
else             → flan-t5-small
embeddings       → all-MiniLM-L6-v2 (both)
```

### Minimal RAG loop

```python
hits = retrieve(query, k=3)
prompt = build_rag_prompt(query, hits)
#answer = generator(prompt)[0]["generated_text"]
answer = generate(prompt) #updated
```
---

**Next notebook:** [`03_advanced_rag_reranking.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/04_rag_systems/03_advanced_rag_reranking.ipynb))  
Hybrid Search · Reranking · RAG Evaluation

---

**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Science/Analytics and ML/AI related opportunities.